In [0]:
#----------------------------------------------------#
# FATO ORDER                                         #
#----------------------------------------------------#

caminho_fato_order = "zeta_project.gold.fact_orders"


#CRIA FATO CASO NÃO EXISTA
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {caminho_fato_order} (
    order_id STRING,
    customer_id STRING,
    order_date DATE,
    order_ts TIMESTAMP,
    gross_amount DECIMAL(18,2),
    discount_total DECIMAL(18,2),
    net_amount DECIMAL(18,2),
    payment_method STRING,
    status_final STRING,
    carrier STRING,
    shipping_cost DECIMAL(18,2),
    shipped_ts TIMESTAMP,
    delivered_ts TIMESTAMP,
    delivery_time_hours DOUBLE,
    is_late INT
)
USING DELTA
CLUSTER BY (order_id, customer_id)
""")

#INSERE NOVOS DADOS
spark.sql(f"""
    INSERT INTO {caminho_fato_order}
    SELECT 
        o.order_id, 
        o.customer_id, 
        to_date(o.order_ts) AS order_date, 
        o.order_ts,
        SUM(i.quantity * i.unit_price) AS gross_amount,
        SUM(i.discount_amount) AS discount_total,
        SUM((i.quantity * i.unit_price) - i.discount_amount) AS net_amount,
        o.payment_method,
        s.delivery_status AS status_final,
        s.carrier, 
        s.shipping_cost, 
        s.shipped_ts, 
        s.delivered_ts,
        CASE 
            WHEN s.shipped_ts IS NOT NULL AND s.delivered_ts IS NOT NULL 
            THEN timestampdiff(HOUR, s.shipped_ts, s.delivered_ts) 
            ELSE 0
        END AS delivery_time_hours,
        CASE 
            WHEN timestampdiff(HOUR, s.shipped_ts, s.delivered_ts) > 72 THEN 1 
            ELSE 0 
        END AS is_late


    FROM zeta_project.silver.orders o

    INNER JOIN zeta_project.silver.customers c 
        ON c.customer_id = o.customer_id
    INNER JOIN zeta_project.silver.shipments s 
        ON s.order_id = o.order_id
    INNER JOIN zeta_project.silver.orders_items i 
        ON i.order_id = o.order_id

    GROUP BY 
        o.order_id, 
        o.customer_id, 
        order_date,
        o.order_ts, 
        o.payment_method,
        status_final, 
        s.carrier, 
        s.shipping_cost, 
        s.shipped_ts, 
        s.delivered_ts
    """)

In [0]:
#----------------------------------------------------#
# FATO ORDER_ITEMS                                   #
#----------------------------------------------------#

caminho_fato_order_items = "zeta_project.gold.fact_order_items"


#CRIA FATO CASO NÃO EXISTA

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {caminho_fato_order_items} (
        order_id STRING,
        product_id STRING,
        quantity LONG,
        unit_price DECIMAL(18,2),
        discount_amount DECIMAL(18,2),
        item_net_amount DECIMAL(18,2)
    ) USING DELTA 
    CLUSTER BY (order_id)
""")

#INSERE NOVOS DADOS
spark.sql(f"""
    INSERT INTO {caminho_fato_order_items}
    SELECT 
        order_id, 
        product_id, 
        SUM(quantity) as quantity, 
        MAX(unit_price) as unit_price,              
        SUM(discount_amount) as discount_amount,    
        SUM((unit_price * quantity) - discount_amount) as item_net_amount 
    FROM zeta_project.silver.orders_items
    GROUP BY order_id, product_id
""")